# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

윈도우 데스크탑의 RTX 5060 ti GPU 환경에서 개발되었습니다.

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- ipykernel 설치
- 아래 셀 다시 실행 : 무한 로딩 시 restart
- hello 출력시 torch 설치

In [1]:
print('hello123')

hello123


In [1]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

Looking in indexes: https://download.pytorch.org/whl/nightly/cu128


In [3]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

2.11.0+cu128
True
NVIDIA GeForce RTX 5060 Ti


In [12]:
!pip install easyocr

     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ---------------------------------------- 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------  2.9/2.9 MB 92.2 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 60.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/311.8 kB ? eta -:--:--
   ---------------------------------------- 311.8/311.8 kB ? eta 0:00:00
   ---------------------------------------- 0.0/43.8 MB ? eta -:--:--
   --- ------------------------------------ 4.4/43.8 MB 140.7 MB/s eta 0:00:01
   -------- ------------------------------- 9.5/43.8 MB 121.3 MB/s eta 0:00:01
   ------------- -------------------------- 14.4/43.8 MB 93.9 MB/s eta 0:00:01
   ----------------- ---------------------- 18.6/43.8 MB 93.9 MB/s eta 0:00:01
   --------------------- ------------------ 23.3/43.8 MB 93.9 MB/s eta 0:00:01
   -------------


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade 

  You can safely remove it manually.

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

# 라이브러리, 데이터, 설정

In [7]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv(os.path.join("data", "train.csv"))
test_df  = pd.read_csv(os.path.join("data", "test.csv"))

# 학습데이터 200개만 추출
train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

Device: cuda


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [8]:
# 양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)

# 사전학습 모델
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 양자화 모델로 로드
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

c:\Users\SSAFY\Desktop\AI2_Challenge\baseline\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SSAFY\.cache\huggingface\hub\models--Qwen--Qwen2.5-VL-7B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by defaul

trainable params: 95,178,752 || all params: 8,387,345,408 || trainable%: 1.1348


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [9]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "당신은 최고 수준의 시각적 질의응답(VQA) 전문가입니다. "
    "주어진 이미지와 질문을 아주 자세히 분석한 후, "
    "정답을 반드시 a, b, c, d 중 하나의 소문자 알파벳으로만 출력하세요. 다른 설명은 필요 없습니다."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"[질문]\n{question}\n\n"
        f"[보기]\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "위 보기를 신중하게 평가한 후, 가장 적절한 정답의 알파벳 하나(a, b, c, d 중 하나)만 출력하세요."
    )


# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [13]:
import torchvision.transforms as T
import easyocr

# OCR 인스턴스 (필요시 활성화 위해 초기화, GPU 사용)
# 처음 실행시 모델을 다운로드합니다.
reader = easyocr.Reader(['ko', 'en'], gpu=True)
USE_OCR = False # 테스트 시 True로 변경해서 성능 비교해보세요!

# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train
        
        # 데이터 증강(Augmentation) 설정 - 색상/대비/선명도 위주 (VQA 특성상 좌우반전은 위험)
        self.aug = T.Compose([
            T.ColorJitter(brightness=0.2, contrast=0.2),
            T.RandomAdjustSharpness(sharpness_factor=2, p=0.5)
        ])

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        import os
        img_path = os.path.join("data", row["path"])
        img = Image.open(img_path).convert("RGB")
        
        if self.train:
            img = self.aug(img)

        q = str(row["question"])
        
        # OCR 텍스트 추가 로직 (USE_OCR=True 일 때만 작동)
        if USE_OCR:
            import numpy as np
            ocr_result = reader.readtext(np.array(img), detail=0)
            if ocr_result:
                q = f"[이미지 내 텍스트: {' '.join(ocr_result)}]\n" + q

        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.1% Complete

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [14]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# 데이터로더
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, collate_fn=DataCollator(processor, True), num_workers=0)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [15]:
from tqdm.auto import tqdm

model = model.to(device)
GRAD_ACCUM = 4

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
num_training_steps = 1 * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 스케일러
scaler = torch.cuda.amp.GradScaler(enabled=True)

# 학습 루프
global_step = 0
for epoch in range(1):
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

# 모델 저장
SAVE_DIR = "qwen2_5_vl_7b_lora"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_10852\609017225.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)
Epoch 1 [train]:   0%|          | 0/180 [00:00<?, ?batch/s]


FileNotFoundError: [Errno 2] No such file or directory: 'train/train_2851.jpg'

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [16]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last

    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

# 앙상블을 위한 3가지 프롬프트 변형
def build_mc_prompt_var1(q, a, b, c, d):
    return f"{q}\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n정답의 알파벳만 말해."
def build_mc_prompt_var2(q, a, b, c, d):
    return f"이미지를 보고 질문에 답하세요.\n질문: {q}\n선택지:\na) {a}\nb) {b}\nc) {c}\nd) {d}\n\n가장 정확한 답을 a,b,c,d 중 하나로 답하시오."
def build_mc_prompt_var3(q, a, b, c, d):
    return f"Question: {q}\nOptions:\n(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\nSelect the correct option letter (a, b, c, or d)."

# 추론을 위해 모든 레이어 활성화
model.eval()
preds = []

from collections import Counter

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    import os
    img = Image.open(os.path.join("data", row["path"])).convert("RGB")
    
    q = str(row["question"])
    if USE_OCR:
        import numpy as np
        ocr_result = reader.readtext(np.array(img), detail=0)
        if ocr_result:
            q = f"[이미지 내 텍스트: {' '.join(ocr_result)}]\n" + q

    # 3개의 프롬프트 생성
    prompts = [
        build_mc_prompt_var1(q, row["a"], row["b"], row["c"], row["d"]),
        build_mc_prompt_var2(q, row["a"], row["b"], row["c"], row["d"]),
        build_mc_prompt_var3(q, row["a"], row["b"], row["c"], row["d"])
    ]
    
    votes = []
    for p in prompts:
        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":p}
            ]}
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=2, do_sample=False,
                                     eos_token_id=processor.tokenizer.eos_token_id)
        output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
        votes.append(extract_choice(output_text))
    
    # 다수결 투표
    majority_vote = Counter(votes).most_common(1)[0][0]
    preds.append(majority_vote)

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
SUBMISSION_PATH = "submission_7b.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
print("Saved", SUBMISSION_PATH)


Inference:   0%|          | 0/6714 [00:00<?, ?sample/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
c:\Users\SSAFY\Desktop\AI2_Challenge\baseline\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(
Inference: 100%|██████████| 6714/6714 [2:38:38<00:00,  1.42s/sample]  

Saved submission_7b.csv


In [17]:
# 모델 응답 예시
print(output_text)

system
당신은 최고 수준의 시각적 질의응답(VQA) 전문가입니다. 주어진 이미지와 질문을 아주 자세히 분석한 후, 정답을 반드시 a, b, c, d 중 하나의 소문자 알파벳으로만 출력하세요. 다른 설명은 필요 없습니다.
user
Question: 이 사진에 보이는 편의점의 이름은 무엇인가요?
Options:
(a) CU
(b) MINISTOP
(c) GS25
(d) 7-ELEVEN
Select the correct option letter (a, b, c, or d).
assistant
d
